# Tool Spec Expansion Generator

이 노트북은 소스 Python 파일을 읽고 LLM(Test용은 gemini로)을 사용하여 구조화된 정보를 추출함으로써, 각 도구(tool)에 대한 `spec_expansion`을 생성

## Setup

1. `GEMINI_API_KEY` 환경 변수를 설정
2. Google GenerativeAI를 설치
   `pip install google-generativeai`

## Process

1. `biomni/tool/` directory 내의 각 `.py` 파일을 읽기
2. AST(Abstract Syntax Tree)를 사용하여 함수들을 추출
3. 각 함수에 대해, 해당 함수의 docstring과 코드를 LLM에 전달
4. Gemini가 `spec_expansion` JSON을 생성
5. 생성된 결과들을 병합하여 `biomni/tool/tool_description_v1/`에 저장

In [139]:
import os
import json
import re
import ast
import time
import pprint
import google.generativeai as genai

from google.colab import userdata


# Configuration
SOURCE_DIR = "" # 직접 경로 수정할 것.
TARGET_DIR = "tool_description_v1/" # 직접 경로 수정할 것.

# RATE LIMIT SETTINGS
REQUEST_DELAY = 60

DOMAINS = [
    "biochemistry", "bioengineering", "bioimaging", "biophysics",
    "cancer_biology", "cell_biology", "genetics", "genomics",
    "glycoengineering", "immunology", "lab_automation", "literature",
    "microbiology", "molecular_biology", "pathology", "pharmacology",
    "physiology", "protocols", "synthetic_biology", "systems_biology",
    "support_tools", "database"
]


genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
model = genai.GenerativeModel("gemini-2.0-flash-lite")

## 1. Extract Functions from Source Code

각 .py 파일을 파싱하여, 개별 함수와 해당 함수의 docstring 및 코드를 추출

In [140]:
def extract_functions_from_source(source_code):
    """Parse source code and extract functions with docstrings and code."""
    tree = ast.parse(source_code)
    functions = {}

    for node in tree.body:
        if isinstance(node, ast.FunctionDef):
            func_name = node.name
            docstring = ast.get_docstring(node)
            func_code = extract_function_code(source_code, func_name, node.lineno)

            functions[func_name] = {
                "docstring": docstring,
                "code": func_code
            }

    return functions


def extract_function_code(source_code, func_name, start_line):
    """Extract function code from source by line number."""
    lines = source_code.split('\n')
    code_lines = [lines[start_line - 1]]  # def line

    # Find function end by indentation
    base_indent = len(lines[start_line - 1]) - len(lines[start_line - 1].lstrip())

    for i in range(start_line, len(lines)):
        line = lines[i]
        if line.strip() == "":
            code_lines.append(line)
            continue

        current_indent = len(line) - len(line.lstrip())
        if current_indent <= base_indent:
            break

        code_lines.append(line)

    return '\n'.join(code_lines)

## 2. Generate spec_expansion with LLM

각 함수에 대해 해당 함수의 docstring과 코드를 LLM에 전달하고, structured information를 추출

In [141]:
SPEC_EXPANSION_PROMPT_SINGLE = """
You are analyzing a single Python function.

Task: Extract "spec_expansion" for this function ONLY.

Function: {func_name}

Docstring:
{docstring}

Code:
{code}

STRICT RULES:
1. ONLY use information from docstring and code above
2. "do_not" add external information or assumptions
3. If a field cannot be extracted, use None

Output ONLY valid JSON in this exact format:
{{
    "return_schema": {{
        "type": "str | dict | list | numpy.ndarray | pandas.DataFrame | ...",
        "description": "..."
    }},
    "tool_type": {{
        "category": "analysis | retrieval | simulation | transformation",
        "domain": "{domain}" # Domain from source filename (e.g., "biochemistry", "genetics")
    }},
    "failure_pattern": {{
        "important": [
            {{
                "condition": "...",
                "source": "Copy EXACT error message from code (e.g., return 'Error: invalid input', raise ValueError('...'))",
                "resolution": "..."
            }}
        ],
        "do_not": []
    }},
    "test_example": None
}}

Category mapping:
- analyze/calculate/compute/predict/estimate → analysis
- search/retrieve/get/find/query → retrieval
- simulate/model → simulation
- convert/transform/liftover → transformation
- others → general

If Returns section is missing: "return_schema": None
If no error handling: "failure_pattern": {{"important": [], "do_not": []}}

EXTRACTION PRIORITY:
1. First, check Returns section in docstring for return_schema
2. Then, check docstring for error descriptions in failure_pattern
3. Finally, check actual code for error handling if docstring is incomplete
"""


def get_spec_expansion_for_function(func_name, docstring, code, domain):
    """Generate spec_expansion for a single function using Gemini."""
    prompt = SPEC_EXPANSION_PROMPT_SINGLE.format(
        func_name=func_name,
        docstring=docstring or "No docstring",
        code=code,
        domain=domain
    )

    # For Gemini
    response = model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(
            temperature=0.1,
            max_output_tokens=4096
        )
    )

    json_str = response.text.strip()

    # Clean up JSON if LLM added extra text
    if json_str.startswith('```json'):
        json_str = json_str[7:]
    if json_str.startswith('```'):
        json_str = json_str[3:]
    if json_str.endswith('```'):
        json_str = json_str[:-3]
    json_str = json_str.strip()

    return json.loads(json_str)

## 3. Process Single Domain

소스 파일을 읽어 모든 함수를 추출한 뒤, 각 함수에 대해 spec_expansion을 생성

In [142]:
def process_domain(domain):
    """Process all functions in a single domain."""
    source_path = os.path.join(SOURCE_DIR, f"{domain}.py")

    # Read source file
    with open(source_path, 'r') as f:
        source_code = f.read()

    print(f"\nProcessing {domain}...")

    # Extract functions
    functions = extract_functions_from_source(source_code)
    print(f"  Found {len(functions)} functions")

    # Generate spec_expansion for each function
    spec_expansion_dict = {}
    for func_name, func_data in functions.items():
        try:
            spec_exp = get_spec_expansion_for_function(
                func_name,
                func_data["docstring"],
                func_data["code"],
                domain
            )
            spec_expansion_dict[func_name] = spec_exp
            print(f"  ✓ {func_name}")
        except Exception as e:
            print(f"  ✗ {func_name}: {e}")

    return spec_expansion_dict

## 4. Merge and Save to tool_description_v1

생성된 `spec_expansion`을 기존의 `tool_description` 파일에 삽입


In [144]:
def merge_and_save(domain, spec_expansion_dict):
    """Merge spec_expansion into tool_description_v2 file and save, preserving formatting."""
    target_path = os.path.join(TARGET_DIR, f"{domain}.py")

    # Read the original source code
    with open(target_path, 'r') as f:
        source_code = f.read()

    # Parse the AST to find the 'description' assignment
    tree = ast.parse(source_code)
    description_assign_node = None

    for node in ast.walk(tree):
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id == 'description':
                    description_assign_node = node
                    break
            if description_assign_node:
                break

    if not description_assign_node:
        print(f"  ✗ Error: 'description' variable not found in {target_path}")
        return

    # Extract the original list using literal_eval
    description_list = ast.literal_eval(description_assign_node.value)

    # Add spec_expansion to each tool and clean description strings
    for tool in description_list:
        tool_name = tool.get('name')
        if tool_name in spec_expansion_dict:
            tool['spec_expansion'] = spec_expansion_dict[tool_name]
            print(f"  ✓ Added spec_expansion to {tool_name}")

            # Ensure 'description' strings are single-line logically by collapsing whitespace
            if 'description' in tool and isinstance(tool['description'], str):
                tool['description'] = ' '.join(tool['description'].split())

            # Also for optional_parameters and required_parameters
            for param_list_key in ['optional_parameters', 'required_parameters']:
                if param_list_key in tool and isinstance(tool[param_list_key], list):
                    for param in tool[param_list_key]:
                        if 'description' in param and isinstance(param['description'], str):
                            param['description'] = ' '.join(param['description'].split())

    # Manually format the list to ensure desired indentation and avoid backslashes
    formatted_tools_strings = []
    for tool_dict in description_list:
        # Use pprint for each dictionary to get a well-indented representation
        # Use a reasonable width to allow implicit string concatenation for long values
        formatted_tool_str = pprint.pformat(tool_dict, indent=4, width=80)
        formatted_tools_strings.append(formatted_tool_str)

    # Reconstruct the 'description =' assignment block
    new_assignment_block = "description = [\n"
    new_assignment_block += ",\n".join(formatted_tools_strings)
    new_assignment_block += "\n]\n"

    # Get the original lines of the file, preserving line endings
    original_lines = source_code.splitlines(keepends=True)

    # Determine the lines to replace based on AST node's lineno/end_lineno
    start_idx = description_assign_node.lineno - 1 # 0-indexed
    end_idx = description_assign_node.end_lineno # This is 1-indexed, so it's the index AFTER the last line

    # Replace the old assignment block with the new one
    output_lines = original_lines[:start_idx] + [new_assignment_block] + original_lines[end_idx:]
    output_code = "".join(output_lines)

    # Save
    with open(target_path, 'w') as f:
        f.write(output_code)

    print(f"  ✓ Saved to tool_description_v2/{domain}.py")

## 5. Run for All Domains

모든 도메인을 처리하여 `spec_expansion`을 생성

In [145]:
def main():
    """Process all domains."""
    for i, domain in enumerate(DOMAINS):
        try:
            print(f"\n{'='*60}")
            print(f"Domain {i+1}/{len(DOMAINS)}: {domain}")
            print(f"{'='*60}")

            # Process domain
            spec_expansion_dict = process_domain(domain)

            # Merge and save
            if spec_expansion_dict:
                merge_and_save(domain, spec_expansion_dict)

            # Sleep between domains
            time.sleep(REQUEST_DELAY)
        except Exception as e:
            print(f"  ✗ Failed: {e}")

    print("\n{'='*60}")
    print("All domains processed")
    print(f"{'='*60}")

# Uncomment to run
main()


Domain 1/22: biochemistry

Processing biochemistry...
  Found 6 functions
  ✓ analyze_circular_dichroism_spectra
  ✓ analyze_rna_secondary_structure_features
  ✓ analyze_protease_kinetics
  ✓ analyze_enzyme_kinetics_assay
  ✓ analyze_itc_binding_thermodynamics
  ✓ analyze_protein_conservation
  ✓ Added spec_expansion to analyze_circular_dichroism_spectra
  ✓ Added spec_expansion to analyze_rna_secondary_structure_features
  ✓ Added spec_expansion to analyze_protease_kinetics
  ✓ Added spec_expansion to analyze_enzyme_kinetics_assay
  ✓ Added spec_expansion to analyze_itc_binding_thermodynamics
  ✓ Added spec_expansion to analyze_protein_conservation
  ✓ Saved to tool_description_v2/biochemistry.py

Domain 2/22: bioengineering

Processing bioengineering...
  Found 7 functions
  ✓ analyze_cell_migration_metrics
  ✓ perform_crispr_cas9_genome_editing
  ✓ analyze_calcium_imaging_data
  ✓ analyze_in_vitro_drug_release_kinetics
  ✓ analyze_myofiber_morphology
  ✓ decode_behavior_from_neural

## 6. Test with Single Domain (Optional)

전체를 실행하기 전에 단일 도메인으로 먼저 프로세스를 테스트


In [146]:
# # Test with a single domain
# test_domain = "biochemistry"  # Change to any domain

# print(f"Starting test for {test_domain}...")
# print(f"Delay: {REQUEST_DELAY}s between requests\n")

# spec_expansion_dict = process_domain(test_domain)
# if spec_expansion_dict:
#     merge_and_save(test_domain, spec_expansion_dict)
#     print(f"\nTest completed for {test_domain}")

# 8. textify_api_dict Test
Verify that the textify_api_dict function correctly fetches the data.

In [149]:
# Biomni github에 있는 실제 함수
# 추가한 버전

def textify_api_dict(api_dict):
    """Convert a nested API dictionary to a nicely formatted string."""
    lines = []
    for category, methods in api_dict.items():
        lines.append(f"Import file: {category}")
        lines.append("=" * (len("Import file: ") + len(category)))
        for method in methods:
            lines.append(f"Method: {method.get('name', 'N/A')}")
            lines.append(f"  Description: {method.get('description', 'No description provided.')}")

            # Process required parameters
            req_params = method.get("required_parameters", [])
            if req_params:
                lines.append("  Required Parameters:")
                for param in req_params:
                    param_name = param.get("name", "N/A")
                    param_type = param.get("type", "N/A")
                    param_desc = param.get("description", "No description")
                    param_default = param.get("default", "None")
                    lines.append(f"    - {param_name} ({param_type}): {param_desc} [Default: {param_default}]")

            # Process optional parameters
            opt_params = method.get("optional_parameters", [])
            if opt_params:
                lines.append("  Optional Parameters:")
                for param in opt_params:
                    param_name = param.get("name", "N/A")
                    param_type = param.get("type", "N/A")
                    param_desc = param.get("description", "No description")
                    param_default = param.get("default", "None")
                    lines.append(f"    - {param_name} ({param_type}): {param_desc} [Default: {param_default}]")

            # Process spec_expansion
            spec_expansion = method.get("spec_expansion")
            if spec_expansion:
                lines.append("  Spec Expansion:")
                # Return Schema
                return_schema = spec_expansion.get("return_schema")
                if return_schema:
                    lines.append(f"    Return Schema Type: {return_schema.get('type', 'N/A')}")
                    lines.append(f"    Return Schema Description: {return_schema.get('description', 'No description')}")
                # Tool Type
                tool_type = spec_expansion.get("tool_type")
                if tool_type:
                    lines.append(f"    Tool Type Category: {tool_type.get('category', 'N/A')}")
                    lines.append(f"    Tool Type Domain: {tool_type.get('domain', 'N/A')}")
                # Failure Pattern
                failure_pattern = spec_expansion.get("failure_pattern")
                if failure_pattern:
                    important_failures = failure_pattern.get("important", [])
                    if important_failures:
                        lines.append("    Failure Pattern (Important):")
                        for fail in important_failures:
                            lines.append(f"      - Condition: {fail.get('condition', 'N/A')}")
                            lines.append(f"        Source: {fail.get('source', 'N/A')}")
                            lines.append(f"        Resolution: {fail.get('resolution', 'N/A')}")
                    do_not_failures = failure_pattern.get("do_not", [])
                    if do_not_failures:
                        lines.append("    Failure Pattern (Do Not):")
                        for fail in do_not_failures:
                            lines.append(f"      - Condition: {fail.get('condition', 'N/A')}")
                            lines.append(f"        Source: {fail.get('source', 'N/A')}")
                            lines.append(f"        Resolution: {fail.get('resolution', 'N/A')}")
                # Test Example
                test_example = spec_expansion.get("test_example")
                if test_example:
                    lines.append(f"    Test Example: {test_example}")

            lines.append("")  # Empty line between methods
        lines.append("")  # Extra empty line after each category

    return "\n".join(lines)



In [150]:
def get_and_display_tool_description(domain_name):
    """Reads the tool description for a given domain and displays it using textify_api_dict.

    Args:
        domain_name (str): The name of the domain (e.g., 'biochemistry').
    """
    target_path = os.path.join(TARGET_DIR, f"{domain_name}.py")

    try:
        # Read and execute the file to get the 'description' list
        namespace = {}
        with open(target_path, 'r') as f:
            exec(f.read(), namespace)

        description_list = namespace.get('description', [])

        # Format the description list into the expected api_dict structure
        api_dict = {domain_name: description_list}

        # Display using textify_api_dict
        print(textify_api_dict(api_dict))

    except FileNotFoundError:
        print(f"Error: File not found at {target_path}. Please ensure the domain has been processed.")
    except Exception as e:
        print(f"An error occurred: {e}")

# Example usage for biochemistry domain
get_and_display_tool_description('biochemistry')

Import file: biochemistry
Method: analyze_circular_dichroism_spectra
  Description: Analyzes circular dichroism (CD) spectroscopy data to determine secondary structure and thermal stability.
  Required Parameters:
    - sample_name (str): Name of the biomolecule sample (e.g., "Znf706", "G-quadruplex") [Default: None]
    - sample_type (str): Type of biomolecule ("protein" or "nucleic_acid") [Default: None]
    - wavelength_data (list or numpy.ndarray): Wavelength values in nm for CD spectrum [Default: None]
    - cd_signal_data (list or numpy.ndarray): CD signal intensity values (typically in mdeg or Δε) [Default: None]
  Optional Parameters:
    - temperature_data (list or numpy.ndarray): Temperature values (°C) for thermal denaturation experiment [Default: None]
    - thermal_cd_data (list or numpy.ndarray): CD signal values at specific wavelength across different temperatures [Default: None]
    - output_dir (str): Directory to save result files [Default: ./]
  Spec Expansion:
    R